# 02 · Ingeniería de Características y Modelado de Clustering
> **Pipeline:** Carga → Pre-filtro → Selección variables (3 capas) → K-Prototypes → Validación → DBSCAN+Gower → Exportación

## 0. Configuración e imports

In [ ]:
import subprocess, sys

# Instalar dependencias adicionales si no están disponibles
for pkg in ['phik', 'kmodes', 'gower', 'plotnine']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import gower
import phik
from phik.report import plot_correlation_matrix
from plotnine import ggplot, aes, geom_line, geom_point, geom_vline, geom_label, labs, xlab, ylab, theme_minimal
import plotnine
from scipy.stats import chi2_contingency
from sklearn.cluster import DBSCAN
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    silhouette_score, davies_bouldin_score
)
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    LabelEncoder, OneHotEncoder, OrdinalEncoder,
    PowerTransformer, StandardScaler
)
from kmodes.kprototypes import KPrototypes

pd.set_option('display.max_columns', None)

def _gower(df, **kw):
    """Wrapper compatible con pandas StringDtype (pandas 2+): convierte a object antes de gower."""
    d = df.copy()
    sc = d.select_dtypes(include='string').columns
    if len(sc):
        d[sc] = d[sc].astype(object)
    return gower.gower_matrix(d, **kw)

print('Imports OK')

## 1. Carga de datos

In [ ]:
df = pd.read_csv('data_limpia.csv')
df.columns = df.columns.str.strip()
df_modelo  = df.copy()

print(f'Shape: {df.shape}')
print(f'Columnas: {list(df.columns)}')
print()
df[['Valortotalplan', 'ValorTotal_scaled']].describe().round(2)

## 2. Selección de variables para K-Prototypes
### Sustento matemático en 3 capas: Varianza/Entropía → Correlación → Silhouette LOO

In [ ]:
# ── PRE-FILTRO ─────────────────────────────────────────────────────────────
# Descarta columnas no aptas:
#   1. Identificadores / texto libre
#   2. Varianza cero (Periodicidad = todos 'M')
#   3. Rango_* = buckets de variables ya incluidas
#   4. > 50% NaN — demasiado hueco para clustering confiable
#   5. Redundantes comerciales confirmadas por analisis previo
# ────────────────────────────────────────────────────────────────────────────

# Imputar Estrato con moda antes del filtro (37% NaN, valor socioeconómico válido)
moda_estrato = df['Estrato'].mode()[0]
df['Estrato'] = df['Estrato'].fillna(moda_estrato)
print(f'Estrato imputado con moda={moda_estrato} | NaN restantes: {df["Estrato"].isnull().sum()}')

EXCLUIR = [
    # Identificadores y texto libre
    'Contrato','Nroentidad','Nit','Nombrecompleto','Numeroidentificacion',
    'Fechanacimiento','Fechaingreso','Direccion','Telefono','Correo',
    'Edadesmascotas','Razasmascotas','ListaEdadesMascotas',
    # Varianza cero
    'Periodicidad',
    # Redundantes (buckets o duplicados de otras variables)
    'Rango_antigedad','Rango_afiliados','Rango_edad','Rango_tarifa',
    'Rango_produccion','Rango_mortalidad','Rango_siniestralidad',
    'Valortotalplan',       # duplica ValorTotal_scaled
    'Tiposseguros_ajuste',  # duplica Tiposseguros
    'PromEdadMascotas',     # solo descriptivo
    'Valormensual',         # correlacion >0.85 con ValorTotal_scaled
    'Rentabilidad',         # excluida por decision de negocio
    # NaN > 50%
    'Tienepadres','Tieneesposa','Tienehijos','Tieneperro','Tienegato','Tiposseguros',
    # Geograficas con mejor representacion en REGION
    'Ciudad','CIUDAD_NORM','CIUDAD_STD','DEPARTAMENTO',
    # Alta cardinalidad o no utiles para clustering
    'Entidad','Codigoproducto','Nombreproducto','Contratante',
    'SECTOR_EMPLEADOR','ACTIVIDAD_ECONOMICA','Top_resultados','Antigedad',
]

candidatas = [c for c in df.columns if c not in EXCLUIR]
resumen = pd.DataFrame({
    'dtype'  : df[candidatas].dtypes,
    'pct_nan': (df[candidatas].isnull().mean() * 100).round(1),
    'n_uniq' : df[candidatas].nunique()
}).sort_values('pct_nan')
print(f'Candidatas tras pre-filtro: {len(candidatas)}')
print(resumen.to_string())

In [ ]:
# ── CAPA 1: Varianza y Entropía de Shannon ──────────────────────────────────
# CV < 0.10    → varianza insuficiente (numéricas)
# H_rel < 0.30 → variable demasiado concentrada (categóricas)
# ────────────────────────────────────────────────────────────────────────────

candidatas_ok = [c for c in candidatas if df[c].isnull().mean() < 0.40]
df_kp = df[candidatas_ok].copy()

numericas   = list(df_kp.select_dtypes(exclude='object').columns)
categoricas = list(df_kp.select_dtypes(include='object').columns)

for c in numericas:   df_kp[c] = df_kp[c].fillna(df_kp[c].median())
for c in categoricas: df_kp[c] = df_kp[c].fillna(df_kp[c].mode()[0])
df_kp = df_kp.reset_index(drop=True)

descartar_num, descartar_cat = [], []

print('=' * 62)
print('CAPA 1A — Coeficiente de Variación (Numéricas)')
print('=' * 62)
for col in numericas:
    media = df_kp[col].mean()
    std   = df_kp[col].std()
    cv    = std if abs(media) < 1e-6 else std / abs(media)
    nota  = f'std={std:.3f} (escalada)' if abs(media) < 1e-6 else f'CV={cv:.4f}'
    flag  = 'OK' if cv >= 0.10 else 'BAJA VARIANZA'
    if cv < 0.10: descartar_num.append(col)
    print(f'  {col:25s}: {nota}  {flag}')

print()
print('=' * 62)
print('CAPA 1B — Entropía de Shannon (Categóricas)')
print('=' * 62)
for col in categoricas:
    probs = df_kp[col].value_counts(normalize=True)
    H     = -np.sum(probs * np.log2(probs + 1e-10))
    H_max = np.log2(len(probs)) if len(probs) > 1 else 1
    H_rel = H / H_max
    flag  = 'OK' if H_rel >= 0.30 else 'MUY CONCENTRADA'
    if H_rel < 0.30: descartar_cat.append(col)
    print(f'  {col:25s}: H_rel={H_rel:.3f}  (n_cat={len(probs)})  {flag}')

numericas       = [c for c in numericas   if c not in descartar_num]
categoricas     = [c for c in categoricas if c not in descartar_cat]
columnas_modelo = numericas + categoricas
print(f'\nDescartadas Capa 1 : {descartar_num + descartar_cat}')
print(f'Pasan a Capa 2     : {columnas_modelo}')

In [ ]:
# ── CAPA 2: Correlación entre variables ──────────────────────────────────────
# Detecta redundancia que distorsiona distancias en K-Prototypes
# ────────────────────────────────────────────────────────────────────────────

def cramers_v(x, y):
    tabla = pd.crosstab(x, y)
    chi2  = chi2_contingency(tabla)[0]
    n     = tabla.sum().sum()
    r, k  = tabla.shape
    return np.sqrt((chi2 / n) / min(r - 1, k - 1))

def eta_cuadrado(df_sub, num_col, cat_col):
    gran_media = df_sub[num_col].mean()
    grupos     = [g.values for _, g in df_sub.groupby(cat_col)[num_col]]
    ss_between = sum(len(g) * (g.mean() - gran_media) ** 2 for g in grupos)
    ss_total   = ((df_sub[num_col] - gran_media) ** 2).sum()
    return ss_between / ss_total if ss_total > 0 else 0

print('=' * 62)
print('CAPA 2A — V de Cramér (Categórica vs Categórica)')
print('V > 0.70 REDUNDANTES  |  V > 0.50 CORRELADAS')
print('=' * 62)
for c1, c2 in itertools.combinations(categoricas, 2):
    v    = cramers_v(df_kp[c1], df_kp[c2])
    flag = 'REDUNDANTES' if v > 0.70 else ('CORRELADAS' if v > 0.50 else 'OK')
    print(f'  {c1:20s} vs {c2:20s}: V={v:.3f}  {flag}')

print()
print('=' * 62)
print('CAPA 2B — Correlación Spearman (Numérica vs Numérica)')
print('|rho| > 0.85 REDUNDANTES')
print('=' * 62)
corr_sp = df_kp[numericas].corr(method='spearman')
for c1, c2 in itertools.combinations(numericas, 2):
    r    = corr_sp.loc[c1, c2]
    flag = 'REDUNDANTES' if abs(r) > 0.85 else 'OK'
    print(f'  {c1:25s} vs {c2:25s}: rho={r:.3f}  {flag}')

print()
print('=' * 62)
print('CAPA 2C — Eta cuadrado (Numérica vs Categórica)')
print('n2 > 0.14 FUERTE  |  n2 > 0.06 MEDIA  |  < 0.06 DEBIL')
print('=' * 62)
for num in numericas:
    for cat in categoricas:
        eta2 = eta_cuadrado(df_kp, num, cat)
        flag = 'FUERTE' if eta2 > 0.14 else ('MEDIA' if eta2 > 0.06 else 'DEBIL')
        print(f'  {num:25s} vs {cat:20s}: n2={eta2:.4f}  {flag}')

In [ ]:
# ── CAPA 3: Silhouette Leave-One-Out ─────────────────────────────────────────
# Usa distancia Gower (nativa para datos mixtos)
# UMBRAL 0.05 (era 0.01 — demasiado agresivo, descartaba variables comerciales clave)
#   delta > +0.05  → modelo mejora notablemente sin ella → candidata a eliminar
#   delta < -0.05  → modelo empeora notablemente sin ella → variable esencial
#   delta ≈ 0      → neutral → decidir con contexto de negocio
# ────────────────────────────────────────────────────────────────────────────

UMBRAL_CAPA3 = 0.05  # ajustar si se quiere más/menos conservador

def silhouette_kp(df_sub, cat_cols, k=2, n=2000, seed=42):
    muestra  = df_sub.sample(min(n, len(df_sub)), random_state=seed).reset_index(drop=True)
    cat_pos  = [muestra.columns.get_loc(c) for c in cat_cols if c in muestra.columns]
    kp       = KPrototypes(n_clusters=k, init='Cao', n_init=3, random_state=seed)
    labels   = kp.fit_predict(muestra.to_numpy(), categorical=cat_pos)
    dist_mat = _gower(muestra)
    return silhouette_score(dist_mat, labels, metric='precomputed')

print('Calculando Silhouette BASE...')
score_base = silhouette_kp(df_kp[columnas_modelo], categoricas)
print(f'Silhouette BASE: {score_base:.4f}')
print(f'Umbral descarte: delta > {UMBRAL_CAPA3} (umbral conservador — preserva variables de negocio)\n')

print('=' * 68)
print('CAPA 3 — Leave-One-Out Silhouette')
print('=' * 68)
resultados = []
for var in columnas_modelo:
    restantes = [c for c in columnas_modelo if c != var]
    cats_r    = [c for c in categoricas     if c != var]
    score     = silhouette_kp(df_kp[restantes], cats_r)
    delta     = score - score_base
    veredicto = ('QUITALA   - modelo mejora sin ella' if delta > UMBRAL_CAPA3
                 else 'ESENCIAL  - modelo empeora sin ella' if delta < -UMBRAL_CAPA3
                 else 'NEUTRAL   - conservar por defecto')
    print(f'  Sin {var:25s}: {score:.4f}  delta={delta:+.4f}  {veredicto}')
    resultados.append({'variable': var, 'silhouette_sin': round(score, 4), 'delta': round(delta, 4)})

df_res     = pd.DataFrame(resultados).sort_values('delta')
esenciales = df_res[df_res['delta'] < -UMBRAL_CAPA3]['variable'].tolist()
neutras    = df_res[df_res['delta'].between(-UMBRAL_CAPA3, UMBRAL_CAPA3)]['variable'].tolist()
ruido      = df_res[df_res['delta'] >  UMBRAL_CAPA3]['variable'].tolist()
print(f'\nESENCIALES : {esenciales}')
print(f'NEUTRALES  : {neutras}')
print(f'RUIDO (Capa3, umbral={UMBRAL_CAPA3}): {ruido}')
print(f'\nNOTA: Variables en RUIDO pueden ser forzadas por lógica de negocio en la siguiente celda.')

## 3. Construcción de df_kprototypes

In [ ]:
# ── Construcción de df_kprototypes con variables corregidas ─────────────────
#
# Criterios aplicados:
#   1. Partimos de esenciales + neutras (Capa 3 con umbral 0.05)
#   2. ELIMINAMOS duplicados detectados por PhiK=1.0:
#      - Region_V ≡ REGION   → queda REGION (categórica nativa)
#      - EstadoCivil_V ≡ Estadocivil → queda Estadocivil (categórica nativa)
#   3. FORZAMOS variables comerciales clave del producto PAP:
#      - Producto (tipo de cobertura contratada)
#      - Total_afiliados (tamaño familia asegurada)
#      - TienePadres_V (composición familiar — independiente)
#      - TieneHijos_V + TieneEsposa_V → fusionadas en tiene_nucleo_familiar
#        (PhiK=0.891: colinealidad alta, solo para K-Prototypes)
# ────────────────────────────────────────────────────────────────────────────

FORZAR_INCLUSION  = ['Producto', 'Total_afiliados',
                     'TieneHijos_V', 'TieneEsposa_V', 'TienePadres_V']
ELIMINAR_DUPLICADOS = ['Region_V', 'EstadoCivil_V', 'Act.valor']  # PhiK=1.0 con REGION y Estadocivil

columnas_finales = list(dict.fromkeys(
    [c for c in (esenciales + neutras) if c not in ELIMINAR_DUPLICADOS]
    + [c for c in FORZAR_INCLUSION if c in df.columns]
))

print(f'Columnas finales ({len(columnas_finales)}): {columnas_finales}')
print(f'  Eliminadas (duplicadas PhiK=1.0): {ELIMINAR_DUPLICADOS}')
print(f'  Forzadas por negocio PAP        : {[c for c in FORZAR_INCLUSION if c in df.columns]}')

df_kprototypes = df[columnas_finales].copy()

# Imputación por mediana/moda
for c in df_kprototypes.select_dtypes(exclude='object').columns:
    df_kprototypes[c] = df_kprototypes[c].fillna(df_kprototypes[c].median())
for c in df_kprototypes.select_dtypes(include='object').columns:
    df_kprototypes[c] = df_kprototypes[c].fillna(df_kprototypes[c].mode()[0])

# Guardar indices validos ANTES del dropna para alinear con df original en exportacion
idx_validos = df_kprototypes.dropna().index.tolist()
df_kprototypes = df_kprototypes.loc[idx_validos].reset_index(drop=True)
print(f'Filas originales: {len(df):,} | Filas modelo: {len(df_kprototypes):,} | '
      f'Eliminadas: {len(df) - len(df_kprototypes):,}')

# Variables binarias 0/1 → convertir a categóricas
# Distancia Hamming es más apropiada que Euclidiana para indicadores discretos
BINARIAS_A_CAT = [c for c in ['TienePadres_V', 'TieneEsposa_V', 'TieneHijos_V']
                  if c in df_kprototypes.columns]
for c in BINARIAS_A_CAT:
    df_kprototypes[c] = df_kprototypes[c].astype(str)
print(f'  Binarias → categórico           : {BINARIAS_A_CAT}')
# ── Fusión TieneEsposa_V + TieneHijos_V → tiene_nucleo_familiar ─────────────
# Sustento: PhiK=0.891 — colinealidad que duplica la señal familiar en
# la distancia Hamming de K-Prototypes sin agregar dimensión independiente.
# La fusión ocurre SOLO en df_kprototypes; df conserva ambas variables
# para el análisis descriptivo de NB03.
df_kprototypes['tiene_nucleo_familiar'] = (
    (df_kprototypes['TieneEsposa_V'] == '1') |
    (df_kprototypes['TieneHijos_V']  == '1')
).map({True: 'SI', False: 'NO'})
df_kprototypes = df_kprototypes.drop(columns=['TieneEsposa_V', 'TieneHijos_V'])
print(f'  tiene_nucleo_familiar creada — reemplaza TieneEsposa_V + TieneHijos_V (PhiK=0.891)')
print(f'  Distribución: {df_kprototypes["tiene_nucleo_familiar"].value_counts().to_dict()}')

cat_final = list(df_kprototypes.select_dtypes(include='object').columns)
num_final = list(df_kprototypes.select_dtypes(exclude='object').columns)

# PowerTransformer Yeo-Johnson SOLO para variables continuas
# (no aplica a binarias ya convertidas ni ordinales enteras pequeñas)
EXCLUIR_PT = ['EstadoCivil_V', 'Region_V']  # por si quedan residuales
pt_scalers = {}
for c in num_final:
    if c not in EXCLUIR_PT:
        pt = PowerTransformer(method='yeo-johnson', standardize=True)
        df_kprototypes[c] = pt.fit_transform(df_kprototypes[[c]]).ravel()
        pt_scalers[c] = pt

joblib.dump(pt_scalers, 'power_transformers.pkl')
print(f'PowerTransformers guardados: {list(pt_scalers.keys())}')

catColumnsPos = [df_kprototypes.columns.get_loc(c)
                 for c in df_kprototypes.select_dtypes('object').columns]

print(f'\nVariables finales  : {list(df_kprototypes.columns)}')
print(f'Numericas          : {num_final}')
print(f'Categoricas        : {cat_final}')
print(f'Posiciones cat     : {catColumnsPos}')
print(f'Shape              : {df_kprototypes.shape}')

---
## CHECKPOINT 1 — Después de construcción de df_kprototypes
> Guarda el dataframe preparado, listas de variables y posiciones categóricas.  
> Si necesitas volver a este punto sin re-ejecutar Capas 1-3, ejecuta la celda de **RESTAURAR** debajo.

In [ ]:
strat_col = 'Producto' if 'Producto' in df_kprototypes.columns else cat_final[0]

ckpt1 = {
    'df_kprototypes': df_kprototypes.copy(),
    'cat_final'     : cat_final,
    'num_final'     : num_final,
    'catColumnsPos' : catColumnsPos,
    'strat_col'     : strat_col,
}
joblib.dump(ckpt1, 'ckpt1_variables.pkl')
print(f'CHECKPOINT 1 guardado: ckpt1_variables.pkl')
print(f'  Shape: {df_kprototypes.shape} | cat={cat_final} | num={num_final}')

In [ ]:
# ── CHECKPOINT 1: RESTAURAR ──────────────────────────────────────────────────
# Descomentar y ejecutar para saltar Capas 1-3 y retomar desde aquí
# ─────────────────────────────────────────────────────────────────────────────
# import joblib
# import pandas as pd, numpy as np
# from sklearn.preprocessing import PowerTransformer
# from kmodes.kprototypes import KPrototypes
# import gower
# from sklearn.metrics import silhouette_score
# from sklearn.model_selection import StratifiedShuffleSplit
# from plotnine import ggplot, aes, geom_line, geom_point, geom_vline, geom_label, labs, xlab, ylab, theme_minimal
# import plotnine, matplotlib.pyplot as plt
#
# ckpt1          = joblib.load('ckpt1_variables.pkl')
# df_kprototypes = ckpt1['df_kprototypes']
# cat_final      = ckpt1['cat_final']
# num_final      = ckpt1['num_final']
# catColumnsPos  = ckpt1['catColumnsPos']
# strat_col      = ckpt1['strat_col']
# K_RANGE        = range(2, 11)
# print(f'CHECKPOINT 1 restaurado: shape={df_kprototypes.shape}')
# print(f'  cat={cat_final} | num={num_final}')
print('Celda RESTAURAR lista — descomentar para usarla.')

In [ ]:
# Correlacion PhiK — maneja variables mixtas (num + cat)
# interval_cols especifica columnas continuas para calculo correcto
phik_matrix = df_kprototypes.phik_matrix(interval_cols=num_final)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    phik_matrix,
    annot=True, fmt='.2f', cmap='Blues',
    vmin=0, vmax=1, ax=ax,
    linewidths=0.5, square=True
)
ax.set_title('PhiK Correlation Matrix — Variables seleccionadas', fontsize=13)
plt.xticks(rotation=40, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print('\nPares con correlacion alta (>0.70):')
for _c1 in phik_matrix.columns:
    for _c2 in phik_matrix.index:
        if _c1 < _c2 and phik_matrix.loc[_c2, _c1] > 0.70:
            print(f'  {_c1} vs {_c2}: {phik_matrix.loc[_c2, _c1]:.3f}')

print('\nMatriz PhiK:')
print(phik_matrix.round(3).to_string())

## 4. Modelado K-Prototypes

In [ ]:
# ── Elbow Method — muestra estratificada ────────────────────────────────────
N_ELBOW   = 10_000
K_RANGE   = range(2, 11)
strat_col = 'Producto' if 'Producto' in df_kprototypes.columns else cat_final[0]

sss = StratifiedShuffleSplit(
    n_splits=1, test_size=N_ELBOW / len(df_kprototypes), random_state=42
)
_, idx_e = next(sss.split(df_kprototypes, df_kprototypes[strat_col]))
df_elbow  = df_kprototypes.iloc[idx_e].reset_index(drop=True)
mat_elbow = df_elbow.to_numpy()
print(f'Muestra Elbow: {len(df_elbow):,} ({len(df_elbow)/len(df_kprototypes)*100:.1f}%)')

costs = []
for k in K_RANGE:
    kp = KPrototypes(n_jobs=-1, n_clusters=k, init='Cao', n_init=3, random_state=42)
    kp.fit_predict(mat_elbow, categorical=catColumnsPos)
    costs.append(kp.cost_)
    print(f'  K={k}  costo={kp.cost_:,.0f}')

# Detección automática del codo con segunda derivada
costs_arr    = np.array(costs)
# np.gradient usa diferencia unilateral en extremos: saltamos bordes para evitar artefacto
deriv2_costs = np.gradient(np.gradient(costs_arr))
codo_idx     = 1 + np.argmax(np.abs(deriv2_costs[1:-1]))
codo_k       = list(K_RANGE)[codo_idx]
print(f'\nCodo detectado automáticamente: K={codo_k}')

df_cost = pd.DataFrame({'K': list(K_RANGE), 'Cost': costs})
plotnine.options.figure_size = (9, 5)
print(
    ggplot(df_cost)
    + geom_line(aes(x='K', y='Cost'))
    + geom_point(aes(x='K', y='Cost'))
    + geom_vline(xintercept=codo_k, color='red', linetype='dashed', size=1)
    + geom_label(aes(x='K', y='Cost', label='K'), size=10, nudge_y=500)
    + labs(title=f'Elbow Method — K-Prototypes (muestra 10k) | Codo sugerido: K={codo_k}')
    + xlab('Numero de Clusters K') + ylab('Costo (cost_)') + theme_minimal()
)

In [ ]:
# ── Silhouette por K + Grid Search de Gamma ─────────────────────────────────
# Paso 1: Buscar gamma óptimo con k=codo_k (muestra 3k, rápido)
# Paso 2: Con gamma_optimo fijo, evaluar Silhouette para cada K (muestra 5k)
# ────────────────────────────────────────────────────────────────────────────

N_SIL    = 5_000
N_GAMMA  = 3_000  # muestra más pequeña para grid search de gamma (rapidez)
GAMMAS   = [0.1, 0.2, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0]

# ── Muestra estratificada compartida ─────────────────────────────────────────
sss2 = StratifiedShuffleSplit(
    n_splits=1, test_size=N_SIL / len(df_kprototypes), random_state=42
)
_, idx_s = next(sss2.split(df_kprototypes, df_kprototypes[strat_col]))
df_sil = df_kprototypes.iloc[idx_s].reset_index(drop=True)

# Submuestra para gamma search
df_gamma = df_sil.sample(min(N_GAMMA, len(df_sil)), random_state=42).reset_index(drop=True)

print(f'Muestra Silhouette: {len(df_sil):,} | Muestra Gamma search: {len(df_gamma):,}')
print('Calculando matriz Gower...')
dist_matrix       = _gower(df_sil)
dist_matrix_gamma = _gower(df_gamma)
print('Lista.\n')

# ── PASO 1: Grid Search de Gamma ─────────────────────────────────────────────
print('=' * 60)
print(f'GRID SEARCH GAMMA (k={codo_k}, muestra {N_GAMMA})')
print('=' * 60)
gamma_results = []
for g in GAMMAS:
    kp_g   = KPrototypes(n_clusters=codo_k, init='Cao', n_init=3,
                         gamma=g, random_state=42)
    lbl_g  = kp_g.fit_predict(df_gamma.to_numpy(), categorical=catColumnsPos)
    n_uniq = len(set(lbl_g))
    if n_uniq < 2:
        print(f'  gamma={g:.2f}  → todos en 1 cluster, skip')
        gamma_results.append((g, np.nan))
        continue
    sil_g = silhouette_score(dist_matrix_gamma, lbl_g, metric='precomputed')
    gamma_results.append((g, round(sil_g, 4)))
    print(f'  gamma={g:.2f}  Silhouette={sil_g:.4f}')

df_gamma_res = pd.DataFrame(gamma_results, columns=['gamma', 'silhouette']).dropna()
gamma_optimo = float(df_gamma_res.loc[df_gamma_res['silhouette'].idxmax(), 'gamma'])
print(f'\nGamma óptimo: {gamma_optimo}  (Silhouette={df_gamma_res["silhouette"].max():.4f})')

# ── PASO 2: Silhouette por K con gamma_optimo ─────────────────────────────────
print(f'\n{"=" * 60}')
print(f'SILHOUETTE POR K (gamma={gamma_optimo}, muestra {N_SIL})')
print('=' * 60)
sil_scores = []
for k in K_RANGE:
    kp     = KPrototypes(n_clusters=k, init='Cao', n_init=3, n_jobs=-1,
                         gamma=gamma_optimo, random_state=42)
    labels = kp.fit_predict(df_sil.to_numpy(), categorical=catColumnsPos)
    score  = silhouette_score(dist_matrix, labels, metric='precomputed')
    sil_scores.append(score)
    print(f'  K={k}  Silhouette(Gower)={score:.4f}')

k_optimo = list(K_RANGE)[sil_scores.index(max(sil_scores))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Gamma search
axes[0].plot([g for g, _ in gamma_results if not np.isnan(_)],
             [s for _, s in gamma_results if not np.isnan(s)],
             marker='o', color='darkorange', linewidth=2)
axes[0].axvline(gamma_optimo, color='crimson', linestyle='--',
                label=f'gamma opt={gamma_optimo}')
axes[0].set_xlabel('Gamma'); axes[0].set_ylabel('Silhouette Score (Gower)')
axes[0].set_title(f'Grid Search Gamma (k={codo_k})')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Plot 2: K search
axes[1].plot(list(K_RANGE), sil_scores, marker='o', color='dodgerblue', linewidth=2)
for x, y in zip(K_RANGE, sil_scores):
    axes[1].annotate(f'{y:.3f}', (x, y), textcoords='offset points',
                     xytext=(0, 10), ha='center', fontsize=8)
axes[1].axvline(k_optimo, color='crimson', linestyle='--',
                label=f'K opt={k_optimo}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score (Gower)')
axes[1].set_title(f'Silhouette por K (gamma={gamma_optimo})')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nK óptimo (Silhouette): {k_optimo}  |  Score: {max(sil_scores):.4f}')
print(f'Gamma óptimo: {gamma_optimo}')
print('Confirmar con el codo del Elbow — ambas métricas deben coincidir.')

In [ ]:
# ── Modelo final — entrena sobre todos los registros ────────────────────────
k_optimo = 3  # Forzado: Elbow codo en K=3; gamma_optimo=2.0 calculado con k=3
dfMatrix      = df_kprototypes.to_numpy()
catColumnsPos = [df_kprototypes.columns.get_loc(c)
                 for c in df_kprototypes.select_dtypes('object').columns]

print(f'Entrenando K-Prototypes K={k_optimo} sobre {len(df_kprototypes):,} registros...')
print(f'Gamma óptimo: {gamma_optimo}')

kproto_final = KPrototypes(
    n_clusters=k_optimo, init='Cao', n_init=10, n_jobs=-1,
    gamma=gamma_optimo, random_state=42
)
etiquetas_finales = kproto_final.fit_predict(dfMatrix, categorical=catColumnsPos)
df_kprototypes['Cluster'] = etiquetas_finales

print(f'Gamma real usado  : {kproto_final.gamma:.4f}')
print(f'Costo final       : {kproto_final.cost_:,.0f}')
print('\nDistribución de clusters:')
dist_clusters = df_kprototypes['Cluster'].value_counts().sort_index()
for c, n in dist_clusters.items():
    print(f'  Cluster {c}: {n:,} ({n/len(df_kprototypes)*100:.1f}%)')

---
## CHECKPOINT 2 — Después de K-Prototypes entrenado
> Guarda el modelo entrenado, el dataframe etiquetado, k_optimo y gamma_optimo.  
> Permite retomar la validación sin re-entrenar (el entrenamiento sobre 345k puede tardar varios minutos).

In [ ]:
# ── CHECKPOINT 2: GUARDAR ────────────────────────────────────────────────────
import joblib

ckpt2 = {
    'df_kprototypes': df_kprototypes.copy(),
    'kproto_final'  : kproto_final,
    'k_optimo'      : k_optimo,
    'gamma_optimo'  : gamma_optimo,
    'cat_final'     : cat_final,
    'num_final'     : num_final,
    'catColumnsPos' : catColumnsPos,
}
joblib.dump(ckpt2, 'ckpt2_modelo_kprototypes.pkl')
print(f'CHECKPOINT 2 guardado: ckpt2_modelo_kprototypes.pkl')
print(f'  K={k_optimo} | gamma={gamma_optimo} | shape={df_kprototypes.shape}')
print(f'  Distribución: {df_kprototypes["Cluster"].value_counts().sort_index().to_dict()}')

In [ ]:
# ── CHECKPOINT 2: RESTAURAR ──────────────────────────────────────────────────
# Descomentar y ejecutar para saltar Elbow + Silhouette + Entrenamiento
# ─────────────────────────────────────────────────────────────────────────────
# import joblib, pandas as pd, numpy as np, matplotlib.pyplot as plt
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.linear_model import LogisticRegression
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score, classification_report
# from sklearn.decomposition import PCA
#
# ckpt2          = joblib.load('ckpt2_modelo_kprototypes.pkl')
# df_kprototypes = ckpt2['df_kprototypes']
# kproto_final   = ckpt2['kproto_final']
# k_optimo       = ckpt2['k_optimo']
# gamma_optimo   = ckpt2['gamma_optimo']
# cat_final      = ckpt2['cat_final']
# num_final      = ckpt2['num_final']
# catColumnsPos  = ckpt2['catColumnsPos']
# print(f'CHECKPOINT 2 restaurado: K={k_optimo} | gamma={gamma_optimo}')
# print(f'  Distribución: {df_kprototypes["Cluster"].value_counts().sort_index().to_dict()}')
print('Celda RESTAURAR lista — descomentar para usarla.')

## 5. Validación, importancia de variables y PCA

In [ ]:
# ── Validación supervisada: Random Forest + Regresión Logística ─────────────
X = df_kprototypes.drop('Cluster', axis=1)
y = df_kprototypes['Cluster']

preprocessor = ColumnTransformer(transformers=[
    ('num', 'passthrough',                          num_final),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_final),
], remainder='drop')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Random Forest
print('Entrenando Random Forest...')
modelo_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                                       class_weight='balanced'))
])
modelo_rf.fit(X_train, y_train)
y_pred_rf = modelo_rf.predict(X_test)
print(f'=== RANDOM FOREST — Accuracy: {accuracy_score(y_test, y_pred_rf):.4f} ===')
print(classification_report(y_test, y_pred_rf))

# Regresion Logistica
print('Entrenando Regresion Logistica...')
modelo_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1))
])
modelo_lr.fit(X_train, y_train)
y_pred_lr = modelo_lr.predict(X_test)
print(f'=== REGRESION LOGISTICA — Accuracy: {accuracy_score(y_test, y_pred_lr):.4f} ===')
print(classification_report(y_test, y_pred_lr))

# Guardar modelo RF
joblib.dump(modelo_rf, 'modelo_clusters_rf.pkl')
print('Guardado: modelo_clusters_rf.pkl')

---
## CHECKPOINT 3 — Después de validación supervisada (RF + Regresión Logística)
> Guarda el modelo Random Forest de producción y las métricas de validación.  
> Permite retomar importancia de variables y PCA sin re-entrenar los clasificadores.

In [ ]:
# ── CHECKPOINT 3: GUARDAR ────────────────────────────────────────────────────
import joblib, pandas as pd

ckpt3 = {
    'modelo_rf'     : modelo_rf,
    'modelo_lr'     : modelo_lr,
    'acc_rf'        : accuracy_score(y_test, y_pred_rf),
    'acc_lr'        : accuracy_score(y_test, y_pred_lr),
    'num_final'     : num_final,
    'cat_final'     : cat_final,
}
joblib.dump(ckpt3, 'ckpt3_validacion.pkl')
print(f'CHECKPOINT 3 guardado: ckpt3_validacion.pkl')
print(f'  RF Accuracy={ckpt3["acc_rf"]:.4f} | LR Accuracy={ckpt3["acc_lr"]:.4f}')

# Guardar métricas como CSV para referencia futura
pd.DataFrame([{
    'k_optimo'    : k_optimo,
    'gamma_optimo': gamma_optimo,
    'acc_rf'      : ckpt3['acc_rf'],
    'acc_lr'      : ckpt3['acc_lr'],
    'n_variables' : len(num_final) + len(cat_final),
    'variables'   : str(num_final + cat_final),
}]).to_csv('metricas_validacion.csv', index=False)
print('  Métricas exportadas: metricas_validacion.csv')

In [ ]:
# ============================================================
# FUNCION DE INFERENCIA — Pipeline completo para produccion
# Aplica: imputacion con estadisticas de entrenamiento
#         → PowerTransformer exacto → RF predict
# ============================================================

# Guardar estadisticas de imputacion del training
_medians_train = {c: df_kprototypes[c].median() for c in num_final}
_modes_train   = {c: df_kprototypes[c].mode()[0] for c in cat_final}
joblib.dump({'medians': _medians_train, 'modes': _modes_train},
           'imputation_stats.pkl')
joblib.dump(columnas_finales, 'columnas_finales.pkl')
print('imputation_stats.pkl guardado')
print('columnas_finales.pkl guardado')


def predecir_cluster(df_nuevo,
                     modelo_path='modelo_clusters_rf.pkl',
                     pt_path='power_transformers.pkl',
                     cols_path='columnas_finales.pkl',
                     imp_path='imputation_stats.pkl'):
    """
    Predice cluster de clientes nuevos — pipeline completo.
    df_nuevo: DataFrame con columnas crudas (SIN escalar).
    """
    _modelo  = joblib.load(modelo_path)
    _pt      = joblib.load(pt_path)
    _cols    = joblib.load(cols_path)
    _imp     = joblib.load(imp_path)

    df_inf = df_nuevo[_cols].copy()

    # Imputar con estadisticas fijas del entrenamiento
    for c, med in _imp['medians'].items():
        if c in df_inf.columns:
            df_inf[c] = df_inf[c].fillna(med)
    for c, mod in _imp['modes'].items():
        if c in df_inf.columns:
            df_inf[c] = df_inf[c].fillna(mod)

    # PowerTransformer exacto de entrenamiento
    for c, pt in _pt.items():
        if c in df_inf.columns:
            df_inf[c] = pt.transform(df_inf[[c]]).ravel()

    # Variables binarias → str (igual que entrenamiento)
    for c in df_inf.select_dtypes(include='number').columns:
        if set(df_inf[c].dropna().unique()).issubset({0, 1, '0', '1'}):
            df_inf[c] = df_inf[c].astype(str)

    # Fusión familiar — replica lo de celda [11] NB02
    # El modelo fue entrenado con tiene_nucleo_familiar en vez de
    # TieneEsposa_V + TieneHijos_V; hay que construirla aquí
    if 'TieneEsposa_V' in df_inf.columns or 'TieneHijos_V' in df_inf.columns:
        esposa = df_inf.get('TieneEsposa_V', pd.Series(['0'] * len(df_inf), index=df_inf.index))
        hijos  = df_inf.get('TieneHijos_V',  pd.Series(['0'] * len(df_inf), index=df_inf.index))
        df_inf['tiene_nucleo_familiar'] = (
            (esposa.astype(str) == '1') | (hijos.astype(str) == '1')
        ).map({True: 'SI', False: 'NO'})
        df_inf = df_inf.drop(columns=[
            c for c in ['TieneEsposa_V', 'TieneHijos_V'] if c in df_inf.columns
        ])

    return _modelo.predict(df_inf)


# Metadata del modelo
import json
from datetime import datetime

metadata = {
    'fecha_entrenamiento': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'k_optimo'           : int(k_optimo),
    'gamma_optimo'       : float(gamma_optimo),
    'costo_kprototypes'  : float(kproto_final.cost_),
    'n_registros'        : int(len(df_kprototypes)),
    'columnas_finales'   : columnas_finales,
    'num_final'          : num_final,
    'cat_final'          : cat_final,
    'distribucion_clusters': df_kprototypes['Cluster']
                             .value_counts().sort_index().to_dict()
}
with open('modelo_metadata.json', 'w', encoding='utf-8') as _f:
    json.dump(metadata, _f, indent=2, default=str)
print('modelo_metadata.json guardado')

# Test de la funcion de inferencia
muestra_test = df[columnas_finales].dropna().head(5)
preds_test   = predecir_cluster(muestra_test)
print(f'Test inferencia OK — predicciones: {preds_test}')


## Diagnóstico de clusters: desbalance, Ward y UMAP
> Análisis complementario post-modelo: valida si el desbalance es natural,
> compara con clustering jerárquico y evalúa separación visual con UMAP-Gower.

In [ ]:
# ============================================================
# DIAGNÓSTICO — desbalance natural vs artefacto
# Chi² para categóricas | KS para numéricas | plots
# ============================================================
from scipy.stats import chi2_contingency, ks_2samp

pal_cl = {0: '#2196F3', 1: '#FF5722', 2: '#4CAF50', 3: '#9C27B0'}

# Plots categóricas
ncols = max(len(cat_final), 1)
fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4))
if ncols == 1:
    axes = [axes]
for j, col in enumerate(cat_final):
    ct = pd.crosstab(df_kprototypes['Cluster'], df_kprototypes[col], normalize='index')
    ct.T.plot(kind='bar', ax=axes[j],
              color=[pal_cl.get(i, 'gray') for i in ct.index], alpha=0.8)
    axes[j].set_title(col, fontsize=11)
    axes[j].tick_params(axis='x', rotation=35)
    axes[j].legend(title='Cluster', fontsize=8)
fig.suptitle('Distribución por Cluster — Categóricas', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

# Plots numéricas
fig2, axes2 = plt.subplots(1, len(num_final), figsize=(5 * len(num_final), 4))
if len(num_final) == 1:
    axes2 = [axes2]
for j, col in enumerate(num_final):
    for cl in sorted(df_kprototypes['Cluster'].unique()):
        vals = df_kprototypes.loc[df_kprototypes['Cluster'] == cl, col]
        axes2[j].hist(vals, bins=30, alpha=0.5,
                      color=pal_cl.get(cl, 'gray'), label=f'Cluster {cl}')
    axes2[j].set_title(col, fontsize=11)
    axes2[j].legend(fontsize=8)
fig2.suptitle('Distribución por Cluster — Numéricas', fontsize=13, fontweight='bold')
fig2.tight_layout()
plt.show()

# Tests estadísticos
print('=' * 62)
print('  TEST: diferencias entre clusters')
print(f"  {'Variable':<22} {'Test':<5} {'Estadístico':>14} {'p-value':>12}  Veredicto")
print('-' * 62)
for col in cat_final:
    ct = pd.crosstab(df_kprototypes['Cluster'], df_kprototypes[col])
    chi2, p, *_ = chi2_contingency(ct)
    v = 'DIFERENTE' if p < 0.05 else 'Similar'
    print(f"  {col:<22} {'Chi2':<5} {chi2:>14.1f} {p:>12.2e}  {v}")
for col in num_final:
    grupos = [df_kprototypes.loc[df_kprototypes['Cluster'] == cl, col].dropna()
              for cl in sorted(df_kprototypes['Cluster'].unique())]
    stat, p = ks_2samp(grupos[0], grupos[1]) if len(grupos) >= 2 else (0, 1)
    v = 'DIFERENTE' if p < 0.05 else 'Similar'
    print(f"  {col:<22} {'KS':<5} {stat:>14.3f} {p:>12.2e}  {v}")
print()
print('Todas DIFERENTE -> desbalance NATURAL (estructura real del negocio)')
print('Alguna Similar  -> posible ARTEFACTO del algoritmo')


In [ ]:
# ============================================================
# CLUSTERING JERARQUICO — Average-Gower vs K-Prototypes
# Ward requiere Euclidea; con datos mixtos usamos average+Gower
# ============================================================
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.preprocessing import LabelEncoder as _LE
from sklearn.metrics import silhouette_score as _sil_w, davies_bouldin_score as _db_w

N_WARD  = 2000
strat_w = 'Producto' if 'Producto' in df_kprototypes.columns else cat_final[0]
df_w = (df_kprototypes.drop('Cluster', axis=1)
        .groupby(strat_w, group_keys=False)
        .apply(lambda g: g.sample(
            min(len(g), N_WARD // df_kprototypes[strat_w].nunique()),
            random_state=42))
        .reset_index(drop=True))

print(f'Muestra Ward: {len(df_w):,} filas — calculando Gower...')
dist_w    = _gower(df_w)
dist_cond = squareform(dist_w, checks=False)
Z         = linkage(dist_cond, method='average')
labels_ward = fcluster(Z, k_optimo, criterion='maxclust') - 1

X_w_enc = df_w.copy()
for _c in [c for c in cat_final if c in X_w_enc.columns]:
    X_w_enc[_c] = _LE().fit_transform(X_w_enc[_c].astype(str))
sil_w   = _sil_w(dist_w, labels_ward, metric='precomputed')
db_w    = _db_w(X_w_enc.values, labels_ward)
counts_w = pd.Series(labels_ward).value_counts().sort_index()

print(f'\nRESULTADO Ward K={k_optimo}')
print(f'  Distribucion  : {counts_w.to_dict()}')
print(f'  Ratio min     : {counts_w.min()/counts_w.sum():.3f}')
print(f'  Silhouette    : {sil_w:.4f}')
print(f'  Davies-Bouldin: {db_w:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
dendrogram(Z, ax=axes[0], truncate_mode='lastp', p=30,
           leaf_rotation=45, leaf_font_size=8,
           color_threshold=Z[-1, 2] * 0.6)
axes[0].set_title(f'Dendrograma Average-Gower (n={len(df_w):,})', fontsize=12)
axes[0].set_xlabel('Muestras')
axes[0].set_ylabel('Distancia Gower')

last = Z[-15:, 2][::-1]
axes[1].plot(range(1, len(last)+1), last, 'o-', color='#673AB7', linewidth=2)
axes[1].axvline(k_optimo, color='red', linestyle='--', alpha=0.6,
                label=f'K={k_optimo} actual')
axes[1].set_title('Ultimas 15 fusiones — codo = K optimo')
axes[1].set_xlabel('Numero de clusters')
axes[1].set_ylabel('Distancia de fusion')
axes[1].set_xticks(range(1, len(last)+1))
axes[1].legend()
plt.tight_layout()
plt.savefig('dendrograma_ward.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: dendrograma_ward.png')

In [ ]:
!pip install umap-learn

In [ ]:

# ============================================================
# UMAP vs PCA — Separacion visual con distancia Gower (mixto)
# ============================================================
import subprocess, sys as _sys

try:
    import umap as _umap
    UMAP_OK = True
except ImportError:
    print('umap-learn no encontrado — instalando...')
    _pip = [
        _sys.executable, '-m', 'pip', 'install', '-q', 'umap-learn',
        '--trusted-host', 'pypi.org',
        '--trusted-host', 'pypi.python.org',
        '--trusted-host', 'files.pythonhosted.org',
        '--trusted-host', '172.19.1.4',
    ]
    _res = subprocess.run(_pip, capture_output=True, text=True)
    if _res.returncode == 0:
        import umap as _umap
        UMAP_OK = True
        print('umap-learn instalado correctamente.')
    else:
        UMAP_OK = False
        print('No se pudo instalar umap-learn (proxy corporativo bloquea la descarga).')
        print('Opcion: descarga manual de los wheels en otra maquina y ejecuta:')
        print('  pip install numba llvmlite pynndescent umap-learn --no-index --find-links <carpeta>')
        print(_res.stderr[-500:] if _res.stderr else '')

if UMAP_OK:
    from sklearn.preprocessing import OrdinalEncoder as _OE, StandardScaler as _SS2
    from sklearn.decomposition import PCA as _PCA2
    from sklearn.metrics import silhouette_score as _sil2

    N_UMAP  = 5000
    strat_u = 'Producto' if 'Producto' in df_kprototypes.columns else cat_final[0]

    # Muestreo estratificado
    _df_base = df_kprototypes.drop('Cluster', axis=1)
    _grp_u = (_df_base
              .groupby(strat_u, group_keys=False)
              .apply(lambda g: g.sample(
                  min(len(g), N_UMAP // df_kprototypes[strat_u].nunique()),
                  random_state=42)))
    # pandas 2.0+: la columna de groupby puede perderse tras apply()
    if strat_u not in _grp_u.columns:
        if strat_u in _grp_u.index.names:
            _grp_u = _grp_u.reset_index(level=strat_u)
        else:
            _grp_u[strat_u] = df_kprototypes.loc[_grp_u.index, strat_u].values
    # Recuperar etiquetas ANTES de perder el índice original
    y_u  = df_kprototypes.loc[_grp_u.index, 'Cluster'].values
    df_u = _grp_u.reset_index(drop=True)

    print(f'Calculando Gower para UMAP (n={len(df_u):,})...')
    dist_u   = _gower(df_u)
    reducer  = _umap.UMAP(n_components=2, metric='precomputed',
                          n_neighbors=30, min_dist=0.1, random_state=42)
    emb_umap = reducer.fit_transform(dist_u)
    print('UMAP listo.')

    X_u_enc = df_u.copy()
    _cat_u  = [c for c in cat_final if c in X_u_enc.columns]
    X_u_enc[_cat_u] = _OE(handle_unknown='use_encoded_value',
                           unknown_value=-1).fit_transform(X_u_enc[_cat_u])
    _pca2    = _PCA2(n_components=2, random_state=42)
    _scaler2 = _SS2()
    emb_pca2 = _pca2.fit_transform(_scaler2.fit_transform(X_u_enc.values))
    pca_var  = _pca2.explained_variance_ratio_.sum()

    sil_umap = _sil2(dist_u,   y_u, metric='precomputed')
    sil_pca2 = _sil2(emb_pca2, y_u)

    pal_u = {0: '#2196F3', 1: '#FF5722', 2: '#4CAF50', 3: '#9C27B0'}
    c_u   = [pal_u.get(int(l), 'gray') for l in y_u]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].scatter(emb_pca2[:, 0], emb_pca2[:, 1], c=c_u, alpha=0.4, s=8)
    axes[0].set_title(f'PCA 2D (varianza: {pca_var:.1%})\nSilhouette: {sil_pca2:.4f}')
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')

    axes[1].scatter(emb_umap[:, 0], emb_umap[:, 1], c=c_u, alpha=0.4, s=8)
    axes[1].set_title(f'UMAP 2D Gower (n_neighbors=30)\nSilhouette Gower: {sil_umap:.4f}')
    axes[1].set_xlabel('UMAP-1')
    axes[1].set_ylabel('UMAP-2')

    from matplotlib.patches import Patch
    leg = [Patch(facecolor=pal_u.get(int(cl), 'gray'), label=f'Cluster {cl}')
           for cl in sorted(set(y_u))]
    for ax in axes:
        ax.legend(handles=leg, fontsize=9)
    plt.suptitle('PCA vs UMAP-Gower — Clusters K-Prototypes',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('umap_vs_pca.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\nSilhouette UMAP (Gower) : {sil_umap:.4f}')
    print(f'Silhouette PCA          : {sil_pca2:.4f}')
    print()
    if sil_umap > sil_pca2 + 0.02:
        print('UMAP muestra mayor separacion: la estructura no es lineal.')
        print('PCA subestima la separacion real entre clusters.')
    else:
        print('PCA y UMAP muestran separacion similar.')
        print('El solapamiento visual es real, no artefacto de PCA.')


In [ ]:
# ── CHECKPOINT 3: RESTAURAR ──────────────────────────────────────────────────
# Descomentar y ejecutar para retomar importancia de variables y PCA
# ─────────────────────────────────────────────────────────────────────────────
# import joblib
# ckpt3     = joblib.load('ckpt3_validacion.pkl')
# modelo_rf = ckpt3['modelo_rf']
# modelo_lr = ckpt3['modelo_lr']
# num_final = ckpt3['num_final']
# cat_final = ckpt3['cat_final']
# print(f'CHECKPOINT 3 restaurado: RF={ckpt3["acc_rf"]:.4f} | LR={ckpt3["acc_lr"]:.4f}')
print('Celda RESTAURAR lista — descomentar para usarla.')

In [ ]:
# ── Importancia de variables — Random Forest ─────────────────────────────────
ohe       = modelo_rf.named_steps['preprocessor'].named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(cat_final))
all_cols  = list(num_final) + cat_names

df_imp = (
    pd.DataFrame({'variable': all_cols,
                  'importancia': modelo_rf.named_steps['classifier'].feature_importances_})
    .sort_values('importancia', ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_imp, x='importancia', y='variable', palette='Blues_r')
plt.title('Importancia de variables — Random Forest (Top 20)')
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()
print(df_imp.to_string(index=False))

In [ ]:
# ── PCA 2D: visualizar clusters + Scree plot ─────────────────────────────────
X_kp = df_kprototypes.drop('Cluster', axis=1).copy()
oe   = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_kp[cat_final] = oe.fit_transform(X_kp[cat_final])

pca      = PCA(n_components=2, random_state=42)
X_pca    = pca.fit_transform(X_kp)
pca_full = PCA(random_state=42).fit(X_kp)
cumvar   = np.cumsum(pca_full.explained_variance_ratio_)
n_80     = int((cumvar >= 0.80).argmax()) + 1
n_95     = int((cumvar >= 0.95).argmax()) + 1

df_pca = pd.DataFrame({'PC1': X_pca[:, 0], 'PC2': X_pca[:, 1],
                        'Cluster': y.values})

pal = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for i, cl in enumerate(sorted(df_pca['Cluster'].unique())):
    sub = df_pca[df_pca['Cluster'] == cl]
    ax.scatter(sub['PC1'], sub['PC2'], label=f'Cluster {cl}',
               color=pal[i % len(pal)], alpha=0.35, s=4)
ax.set_title(f'Clusters K-Prototypes — PCA 2D\nVarianza explicada: {pca.explained_variance_ratio_.sum():.1%}')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.legend(markerscale=4)

ax2 = axes[1]
n_comp = len(pca_full.explained_variance_ratio_)
ax2.bar(range(1, n_comp + 1), pca_full.explained_variance_ratio_,
        alpha=0.7, color='steelblue', label='Individual')
ax2.plot(range(1, n_comp + 1), cumvar, 'r-o', ms=5, label='Acumulada')
ax2.axhline(0.80, color='gray',  linestyle='--', alpha=0.6, label='80%')
ax2.axhline(0.95, color='black', linestyle=':',  alpha=0.6, label='95%')
ax2.set_xlabel('Componente Principal')
ax2.set_ylabel('Varianza explicada')
ax2.set_title('Scree Plot')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()
print(f'Varianza PC1+PC2: {pca.explained_variance_ratio_.sum():.1%}')
print(f'Componentes para 80%: {n_80}  |  para 95%: {n_95}')

In [ ]:
# ── Análisis descriptivo de clusters ─────────────────────────────────────────
PAL_CLUSTERS = ['#682F2F','#B9C0C9','#9F8A78','#F3AB60','#98FF98']
n_clusters   = df_kprototypes['Cluster'].nunique()

# Boxplots numéricos por cluster
fig, axes = plt.subplots(1, len(num_final), figsize=(5 * len(num_final), 4))
if len(num_final) == 1:
    axes = [axes]
for ax, col in zip(axes, num_final):
    sns.boxplot(x='Cluster', y=col, data=df_kprototypes,
                palette=PAL_CLUSTERS[:n_clusters], ax=ax)
    ax.set_title(col)
plt.suptitle('Distribucion numerica por Cluster', fontweight='bold')
plt.tight_layout()
plt.show()

# Perfil promedio numérico
print('=== PERFIL NUMERICO POR CLUSTER ===')
print(df_kprototypes.groupby('Cluster')[num_final].mean().round(2).to_string())

# Perfil categórico (top 3 por cluster)
print('\n=== PERFIL CATEGORICO POR CLUSTER ===')
for col in cat_final:
    print(f'\n--- {col} ---')
    print(df_kprototypes.groupby('Cluster')[col]
          .apply(lambda x: x.value_counts(normalize=True).head(3).round(3)))

## 6. Exportación K-Prototypes

In [ ]:
# Agregar Cluster al df original con alineacion correcta de indices
# idx_validos garantiza que el cluster va al cliente correcto aunque
# se hayan eliminado filas en el dropna de celda [11]
import numpy as np

df['Cluster'] = np.nan
df.loc[idx_validos, 'Cluster'] = df_kprototypes['Cluster'].values
df['Cluster'] = df['Cluster'].astype('Int64')  # entero nullable

n_asignados    = df['Cluster'].notna().sum()
n_sin_cluster  = df['Cluster'].isna().sum()
print(f'Clientes con cluster  : {n_asignados:,}')
print(f'Clientes sin cluster  : {n_sin_cluster:,} (NaN por datos incompletos)')
print(f'Distribucion clusters :')
print(df['Cluster'].value_counts().sort_index())

df.to_csv('resultado_etiquetado.csv', index=False)
print(f'\nGuardado resultado_etiquetado.csv  |  Shape: {df.shape}')


## 7. DBSCAN + Distancia de Gower
> **Nota:** La matriz de Gower sobre el dataset completo (>300k filas) requiere ~350 GB RAM.
> Se usa una muestra estratificada de 10,000 registros para ajustar DBSCAN;
> las etiquetas se propagan al dataset completo mediante KNN.
>
> **Variables:** DBSCAN usa las 11 candidatas que pasaron Capas 1+2 — no se restringe
> a las 7 de K-Prototypes porque el criterio 'ruido' de Capa 3 es K-Prototypes-específico.

In [ ]:
# ── Paso 1: Preparación dataset DBSCAN ──────────────────────────────────────
COLS_DB_NUM = ['Edad', 'Estrato', 'Act.valor', 'Cuotas', 'Cantidad_mascotas', 'ValorTotal_scaled']
COLS_DB_CAT = ['Estado', 'Sexo', 'Estadocivil', 'Producto', 'REGION']
COLS_DB     = COLS_DB_NUM + COLS_DB_CAT

df_raw = df[COLS_DB].copy()
for c in df_raw.select_dtypes(exclude='object').columns:
    df_raw[c] = df_raw[c].fillna(df_raw[c].median())
for c in df_raw.select_dtypes(include='object').columns:
    df_raw[c] = df_raw[c].fillna(df_raw[c].mode()[0])
df_raw = df_raw.dropna().reset_index(drop=True)

print(f'Shape DBSCAN: {df_raw.shape}')
print(f'Numericas ({len(COLS_DB_NUM)}): {COLS_DB_NUM}')
print(f'Categoricas ({len(COLS_DB_CAT)}): {COLS_DB_CAT}')
df_raw[COLS_DB_NUM].describe().round(3)

In [ ]:
# ── Paso 2: Muestra estratificada + Matriz de Gower ─────────────────────────
SAMPLE_N = 10_000

# Detectar nombre exacto de la columna Producto (case-insensitive)
STRAT_COL = next((c for c in df_raw.columns if c.lower() == 'producto'), None)
if STRAT_COL is None:
    STRAT_COL = df_raw.select_dtypes(exclude='number').columns[0]
    print(f'Columna Producto no encontrada — usando: {STRAT_COL!r}')
else:
    print(f'Columna de estratificación: {STRAT_COL!r}')

df_sample = (
    df_raw
    .groupby(STRAT_COL, group_keys=False)
    .apply(lambda g: g.sample(
        n=max(1, round(SAMPLE_N * len(g) / len(df_raw))), random_state=42
    ))
    .reset_index(drop=True)
)

# Verificar que la columna de estratificación sigue presente tras el groupby
if STRAT_COL not in df_sample.columns:
    df_sample[STRAT_COL] = df_raw.loc[df_sample.index, STRAT_COL].values

print(f'Muestra: {df_sample.shape}  (estratificada por {STRAT_COL!r})')
print(df_sample[STRAT_COL].value_counts())

cat_mask = np.array([not pd.api.types.is_numeric_dtype(df_sample[c]) for c in df_sample.columns])
print(f'\nCalculando matriz Gower ({df_sample.shape[1]} variables)...')
gower_mat = _gower(df_sample, cat_features=cat_mask)
assert gower_mat.min() >= 0 and gower_mat.max() <= 1.0001, 'Valores fuera de rango [0,1]'
print(f'Shape: {gower_mat.shape}  |  Rango: [{gower_mat.min():.4f}, {gower_mat.max():.4f}]  OK')

In [ ]:
# ── Paso 3: Selección de eps con k-distancia ─────────────────────────────────
# min_samples=5 (estándar para datos de alta dimensión mixta)
# Era n_features*2=22 → demasiado restrictivo → DBSCAN detectaba 1 solo cluster
n_features  = df_sample.shape[1]
min_samples = 5  # fijo conservador — era max(5, n_features*2)=22
print(f'min_samples = 5 (era {max(5, n_features*2)} — reducido para detectar más clusters)')

nbrs = NearestNeighbors(n_neighbors=min_samples, metric='precomputed', n_jobs=-1)
nbrs.fit(gower_mat)
distances, _ = nbrs.kneighbors(gower_mat)
k_dist = np.sort(distances[:, -1])[::-1]

# Detección automática del codo con segunda derivada
deriv2       = np.gradient(np.gradient(k_dist))
codo_idx     = np.argmax(np.abs(deriv2))
eps_sugerido = round(float(k_dist[codo_idx]), 4)
print(f'Codo en índice {codo_idx} → eps sugerido = {eps_sugerido}')

# Rango de eps para la búsqueda posterior (más amplio)
eps_min = round(max(0.10, eps_sugerido * 0.5), 4)
eps_max = round(min(0.60, eps_sugerido * 3.0), 4)
print(f'Rango de búsqueda eps: [{eps_min}, {eps_max}]')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_dist, color='steelblue', linewidth=1.2, label=f'k-distancia Gower (k={min_samples})')
ax.axvline(codo_idx, color='crimson', linestyle='--', linewidth=1.5,
           label=f'Codo → eps={eps_sugerido}')
ax.axhline(eps_sugerido, color='orange', linestyle=':', linewidth=1.2)
ax.set_xlabel('Puntos ordenados')
ax.set_ylabel(f'Distancia al {min_samples}-ésimo vecino')
ax.set_title('Gráfico k-distancia (Gower) — Selección de eps')
ax.legend()
plt.tight_layout()
plt.savefig('k_distancia.png', dpi=150)
plt.show()

In [ ]:
# ── Paso 4: Ajuste DBSCAN + evaluación de calidad ───────────────────────────
def evaluar_dbscan(labels, dist_mat):
    """Silhouette y Davies-Bouldin solo sobre puntos no-outlier."""
    mask = labels != -1
    if mask.sum() < 2 or len(set(labels[mask])) < 2:
        return np.nan, np.nan
    sub_dist = dist_mat[np.ix_(mask, mask)]
    sub_lbls = labels[mask]
    sil = silhouette_score(sub_dist, sub_lbls, metric='precomputed')
    dbi = davies_bouldin_score(sub_dist, sub_lbls)
    return round(sil, 4), round(dbi, 4)

# Búsqueda exhaustiva de eps en rango ampliado (usa eps_min/eps_max de celda anterior)
print('Búsqueda de eps óptimo en rango ampliado...')
eps_candidatos = np.round(np.linspace(eps_min, eps_max, 12), 4)
resultados_eps = []

for eps_c in eps_candidatos:
    lbl_c  = DBSCAN(eps=eps_c, min_samples=min_samples,
                    metric='precomputed', n_jobs=-1).fit_predict(gower_mat)
    n_c    = len(set(lbl_c)) - (1 if -1 in lbl_c else 0)
    n_out  = int((lbl_c == -1).sum())
    pct_out = n_out / len(lbl_c) * 100
    if n_c < 2:
        resultados_eps.append({'eps': eps_c, 'n_clusters': n_c, 'silhouette': np.nan,
                                'davies_bouldin': np.nan, 'pct_outliers': round(pct_out, 1)})
        print(f'  eps={eps_c:.4f}  clusters={n_c}  outliers={pct_out:.1f}%  sil=N/A')
        continue
    s, d = evaluar_dbscan(lbl_c, gower_mat)
    resultados_eps.append({'eps': eps_c, 'n_clusters': n_c, 'silhouette': s,
                            'davies_bouldin': d, 'pct_outliers': round(pct_out, 1)})
    print(f'  eps={eps_c:.4f}  clusters={n_c}  outliers={pct_out:.1f}%  sil={s}  dbi={d}')

df_res_eps = pd.DataFrame(resultados_eps)
print('\nResumen búsqueda:')
print(df_res_eps.to_string(index=False))

# Selección: máximo Silhouette con outliers < 30%
df_validos = df_res_eps[df_res_eps['pct_outliers'] < 30].dropna(subset=['silhouette'])
if len(df_validos) > 0:
    mejor      = df_validos.sort_values('silhouette', ascending=False).iloc[0]
    eps_final  = mejor['eps']
    print(f'\nMejor eps={eps_final}  clusters={int(mejor["n_clusters"])}  '
          f'Silhouette={mejor["silhouette"]}  outliers={mejor["pct_outliers"]}%')
else:
    eps_final = eps_sugerido
    print(f'\nNo se encontró eps con <30% outliers y ≥2 clusters. Usando eps={eps_final}.')

labels_final = DBSCAN(eps=eps_final, min_samples=min_samples,
                      metric='precomputed', n_jobs=-1).fit_predict(gower_mat)
sil_score, dbi_score = evaluar_dbscan(labels_final, gower_mat)
n_clusters_final = len(set(labels_final)) - (1 if -1 in labels_final else 0)
n_outliers_final = int((labels_final == -1).sum())

print(f'\nResultado final DBSCAN:')
print(f'  eps={eps_final}  |  clusters={n_clusters_final}  |  outliers={n_outliers_final} ({n_outliers_final/len(labels_final)*100:.1f}%)')
print(f'  Silhouette={sil_score}  |  Davies-Bouldin={dbi_score}')
print(pd.Series(labels_final).value_counts().sort_index()
      .rename(index=lambda x: f'Cluster {x}' if x >= 0 else 'Outliers (-1)').to_string())

In [ ]:
# ── Paso 5: Visualización PCA 2D ────────────────────────────────────────────
scaler_pca   = StandardScaler()
X_num_scaled = scaler_pca.fit_transform(df_sample[COLS_DB_NUM])
X_pca_db     = PCA(n_components=2, random_state=42).fit_transform(X_num_scaled)
var_exp      = PCA(n_components=2, random_state=42).fit(X_num_scaled).explained_variance_ratio_ * 100

palette   = plt.cm.tab10.colors
color_map = {lbl: ('lightgray' if lbl == -1 else palette[i % len(palette)])
             for i, lbl in enumerate(sorted(set(labels_final)))}

fig, ax = plt.subplots(figsize=(10, 6))
for lbl in sorted(set(labels_final)):
    mask = labels_final == lbl
    ax.scatter(X_pca_db[mask, 0], X_pca_db[mask, 1],
               c=color_map[lbl], s=12,
               alpha=0.4 if lbl == -1 else 0.7,
               label=f'{'Outliers' if lbl == -1 else f'Cluster {lbl}'} (n={mask.sum()})',
               edgecolors='none')
ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}% var)')
ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}% var)')
ax.set_title(f'DBSCAN + Gower — PCA 2D | {len(COLS_DB)} variables | muestra 10k')
ax.legend(markerscale=2, fontsize=9)
plt.tight_layout()
plt.savefig('dbscan_gower_pca.png', dpi=150)
plt.show()

In [ ]:
# ── Paso 6: Perfiles de clusters + Heatmap de distancias ────────────────────
df_sample_lbl = df_sample.copy()
df_sample_lbl['cluster_db'] = labels_final
no_out = df_sample_lbl[df_sample_lbl['cluster_db'] != -1]

# Perfil numerico
perfil_num = (
    no_out.groupby('cluster_db')[COLS_DB_NUM]
    .agg(['mean', 'median']).round(3)
)
perfil_num.columns = ['_'.join(c) for c in perfil_num.columns]

# Perfil categorico (moda)
perfil_cat = (
    no_out.groupby('cluster_db')[COLS_DB_CAT]
    .agg(lambda x: x.mode().iloc[0] if len(x) > 0 else np.nan)
)

perfil_completo = pd.concat([perfil_cat, perfil_num], axis=1)
print('Perfil de clusters DBSCAN:')
display(perfil_completo)
perfil_completo.to_csv('perfiles_clusters.csv')
print('Guardado: perfiles_clusters.csv')

# Heatmap de distancias medias entre clusters
clusters_uniq = sorted(set(labels_final[labels_final != -1]))
if len(clusters_uniq) >= 2:
    dist_entre = pd.DataFrame(index=clusters_uniq, columns=clusters_uniq, dtype=float)
    for ci in clusters_uniq:
        for cj in clusters_uniq:
            idx_i = np.where(labels_final == ci)[0]
            idx_j = np.where(labels_final == cj)[0]
            dist_entre.loc[ci, cj] = gower_mat[np.ix_(idx_i, idx_j)].mean()
    dist_entre = dist_entre.astype(float)
    fig, ax = plt.subplots(figsize=(max(4, len(clusters_uniq)), max(3, len(clusters_uniq))))
    sns.heatmap(dist_entre, annot=True, fmt='.3f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, vmin=0, vmax=1)
    ax.set_title('Distancia media Gower entre clusters DBSCAN')
    plt.tight_layout()
    plt.savefig('heatmap_distancias.png', dpi=150)
    plt.show()
    print('Guardado: heatmap_distancias.png')
else:
    print('Solo 1 cluster — heatmap no aplica. Ajusta eps para obtener mas clusters.')

In [ ]:
# ── Paso 7: Propagación KNN al dataset completo ──────────────────────────────
# DBSCAN no tiene predict() → propagamos con KNN (k = min_samples por coherencia semantica)
le_dict      = {}
X_knn_sample = df_sample[COLS_DB_NUM].values.copy().astype(float)
X_knn_full   = df_raw[COLS_DB_NUM].values.copy().astype(float)

for c in COLS_DB_CAT:
    le = LabelEncoder().fit(df_raw[c].astype(str))
    le_dict[c] = le
    X_knn_sample = np.hstack([X_knn_sample, le.transform(df_sample[c].astype(str)).reshape(-1, 1)])
    X_knn_full   = np.hstack([X_knn_full,   le.transform(df_raw[c].astype(str)).reshape(-1, 1)])

scaler_knn      = StandardScaler()
X_knn_sample_sc = scaler_knn.fit_transform(X_knn_sample)
X_knn_full_sc   = scaler_knn.transform(X_knn_full)

mask_labeled = labels_final != -1
k_knn = int(min_samples)
print(f'KNN propagacion: k={k_knn} (= min_samples de DBSCAN)')

knn_clf = KNeighborsClassifier(n_neighbors=k_knn, n_jobs=-1)
knn_clf.fit(X_knn_sample_sc[mask_labeled], labels_final[mask_labeled])
labels_full = knn_clf.predict(X_knn_full_sc)

print(f'Etiquetas propagadas a {len(labels_full):,} registros')
print(pd.Series(labels_full).value_counts().sort_index()
      .rename(index=lambda x: f'Cluster {x}').to_string())

df_salida = df.copy()
df_salida['cluster_dbscan'] = labels_full
df_salida.to_csv('clientes_segmentados.csv', index=False)
print(f'Guardado: clientes_segmentados.csv  |  Shape: {df_salida.shape}')

## 8. Resumen ejecutivo — Comparación K-Prototypes vs DBSCAN

In [ ]:
kp_labels   = df_kprototypes['Cluster'].values
kp_clusters = len(set(kp_labels))
kp_dist     = pd.Series(kp_labels).value_counts(normalize=True).sort_index() * 100

print('=' * 60)
print('  RESUMEN EJECUTIVO — COMPARACION DE CLUSTERING')
print('=' * 60)
print(f"""
DATASET
  Registros totales     : {len(df_raw):,}

K-PROTOTYPES  (dataset completo — {len(df_kprototypes):,} registros)
  Algoritmo             : K-Prototypes (Cao init)
  Distancia             : Euclidiana (num) + Hamming (cat)
  Transformacion previa : PowerTransformer Yeo-Johnson
  Clusters encontrados  : {kp_clusters}
  Distribucion:
{kp_dist.rename(index=lambda x: f'    Cluster {x}').to_string()}

DBSCAN + GOWER  (muestra estratificada — {len(df_sample):,} registros)
  Algoritmo             : DBSCAN (density-based)
  Distancia             : Gower (mixta, rango [0,1])
  eps seleccionado      : {eps_final}
  min_samples           : {min_samples}
  Clusters encontrados  : {n_clusters_final}
  Outliers en muestra   : {n_outliers_final} ({round(n_outliers_final/len(df_sample)*100,1)}%)
  Silhouette Score      : {sil_score}
  Davies-Bouldin Index  : {dbi_score}

ARCHIVOS GENERADOS
  resultado_etiquetado.csv  — Dataset + Cluster (K-Prototypes)
  modelo_clusters_rf.pkl    — Random Forest entrenado
  k_distancia.png           — Curva k-distancia para eps
  dbscan_gower_pca.png      — PCA 2D clusters DBSCAN
  heatmap_distancias.png    — Distancias Gower entre clusters
  perfiles_clusters.csv     — Perfiles descriptivos DBSCAN
  clientes_segmentados.csv  — Dataset completo + cluster_dbscan
""")